# BirdCLEF+ 2025 - Pretrained models: Perch 2.0 embeddings of ALL train_soundscapes (teacher inputs for a noisy student)

**T4 GPU, frozen feature extraction only.** The files are `sorted(train_soundscapes/*.ogg)`, counted at runtime, and `test_soundscapes` is never read.

**Windows:** non-overlapping 5 s, `n_win = max(1, ceil(len / WIN))`, end padded with zeros, `win` 0-based. This is the same rule and sort as in `ns_a1_all`.

**Perch 2.0:** the same ONNX file, per-window preprocessing, checksum and CUDA check as `soundscape_embed`.

**Decode failures** are recorded in the meta file, and more than 20 stop the run. Each notebook keeps its own list, so the later student must assert that the lists and the two csv files are equal.

**Output:** `perch_all_embeddings.npy` (float32, N x 1536), `soundscape_all_windows.csv` (filename, win), `ns_perch_meta.json`. Rows go to a temporary file as they are produced. The final names only appear after the check against the `soundscape_embed` output has passed, and the meta file is written last.

In [ ]:
import os, time
import torch

T_NOTEBOOK = time.time()
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import subprocess, sys

ORT_VERSION = "1.31.1"          # same as soundscape_embed
if not SMOKE:
    # torch is already imported above, which makes its CUDA libraries available to onnxruntime.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"onnxruntime-gpu=={ORT_VERSION}"], check=True)
    print("torch CUDA:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())

In [ ]:
import gc, glob, hashlib, json, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly
import onnxruntime as ort

# On Kaggle every check is on. M3_STRICT only re-enables named checks inside a local smoke test.
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE or name in STRICT

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (smoke test)"
MAX_FAILED = int(os.environ.get("M3_MAX_FAILED", "20"))
GUARD_AFTER = int(os.environ.get("M3_GUARD_WINDOWS", "500"))
RECHECK_EVERY = int(os.environ.get("M3_GUARD_RECHECK", "5000"))   # read-ahead hides slow decoding at the first check
REF_WINDOWS = 12000
SANITY_LIMIT = 1e-3

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

def list_soundscapes(data_dir):
    """The file order of both notebooks: bare names of train_soundscapes/*.ogg, sorted."""
    names = sorted(p.name for p in (data_dir / "train_soundscapes").glob("*.ogg"))
    if not names:
        raise RuntimeError(f"no .ogg files in {data_dir / 'train_soundscapes'}")
    return names

def add_failure(failed, name, err):
    failed.append((name, err))
    print("decode failure:", name, err, flush=True)
    if len(failed) > MAX_FAILED:
        raise RuntimeError(f"{len(failed)} files failed to decode (limit {MAX_FAILED}), first: {failed[:3]}")

def make_guard(label, total_files):
    """Projects the whole notebook after GUARD_AFTER windows and again every RECHECK_EVERY windows. The rate starts
    after the first GPU batch, so one-off graph compilation does not inflate it."""
    state = dict(mark=None, next_check=GUARD_AFTER)
    def check(n_done, windows_seen, files_ok):
        now = time.time()
        if state["mark"] is None:
            state["mark"] = (n_done, now)
            return
        n0, t0 = state["mark"]
        if n_done - n0 < state["next_check"]:
            return
        state["next_check"] = n_done - n0 + RECHECK_EVERY
        rate = (n_done - n0) / max(now - t0, 1e-6)
        est_total = windows_seen / max(files_ok, 1) * total_files
        elapsed = now - T_NOTEBOOK
        projected = elapsed + max(est_total - n_done, 0) / rate
        msg = (f"{label}: {rate:.1f} windows/s after {n_done} windows, about {est_total:.0f} windows in total, "
               f"{elapsed:.0f}s elapsed -> projected notebook time {projected:.0f}s = {projected / 3600:.2f} h "
               f"(limit {BUDGET_S:.0f}s = {BUDGET_S / 3600:.2f} h)")
        print("TIMING GUARD |", msg, flush=True)
        if projected > BUDGET_S:
            raise RuntimeError(f"Projected GPU time is over the limit. {msg}")
    return check

def load_reference_keys(csv_path, n_rows, dim, known_names):
    """Checks that cost nothing, before any GPU work."""
    ref = pd.read_csv(csv_path)
    if list(ref.columns[:2]) != ["filename", "win"]:
        raise RuntimeError(f"unexpected reference csv columns: {list(ref.columns)}")
    ref = ref[["filename", "win"]].astype({"filename": str, "win": int})
    if ref.duplicated().any():
        raise RuntimeError("duplicate (filename, win) rows in the reference csv")
    if len(ref) != n_rows:
        raise RuntimeError(f"reference csv has {len(ref)} rows, its array has {n_rows}")
    if strict("ref") and n_rows != REF_WINDOWS:
        raise RuntimeError(f"expected {REF_WINDOWS} reference windows, found {n_rows}")
    unknown = sorted(set(ref["filename"]) - set(known_names))
    if unknown:
        raise RuntimeError(f"{len(unknown)} reference files are not in train_soundscapes, first: {unknown[:3]}")
    print(f"reference: {len(ref)} windows, {ref['filename'].nunique()} files, {dim}-d", flush=True)
    return ref

def align_to_reference(ref, windows):
    """Row of the new array for each reference row, matched on (filename, win), in reference row order."""
    merged = ref.merge(windows.assign(new_row=np.arange(len(windows))), on=["filename", "win"], how="left", validate="one_to_one")
    missing = merged[merged["new_row"].isna()]
    if len(missing):
        raise RuntimeError(f"{len(missing)} reference windows have no row in this run, first: "
                           f"{missing[['filename', 'win']].head(3).values.tolist()}")
    return merged["new_row"].astype(np.int64).values

def check_windows_table(windows, names, failed):
    """Sorted file order, windows 0..n-1 per file, and exactly the failed files left out."""
    done = list(dict.fromkeys(windows["filename"].tolist()))
    bad = {n for n, _ in failed}
    if done != [n for n in names if n not in bad]:
        raise RuntimeError("the embedded files are not the sorted file list minus the failed files")
    for name, g in windows.groupby("filename", sort=False):
        assert g["win"].tolist() == list(range(len(g))), f"windows of {name} are not 0..n-1"

def finish_array(bin_path, npy_tmp, n_rows, dim, chunk_rows=4096):
    """Turns the appended raw float32 file into a .npy in chunks, checks every value, and returns its name."""
    expected = n_rows * dim * 4
    if os.path.getsize(bin_path) != expected:
        raise RuntimeError(f"{bin_path} has {os.path.getsize(bin_path)} bytes, expected {expected}")
    mm = np.lib.format.open_memmap(npy_tmp, mode="w+", dtype=np.float32, shape=(n_rows, dim))
    with open(bin_path, "rb") as f:
        for start in range(0, n_rows, chunk_rows):
            part = np.frombuffer(f.read(min(chunk_rows, n_rows - start) * dim * 4), dtype="<f4").reshape(-1, dim)
            if not np.isfinite(part).all():
                raise RuntimeError(f"non-finite values in rows {start}..{start + len(part)}")
            mm[start:start + len(part)] = part
    mm.flush()
    del mm
    gc.collect()
    os.remove(bin_path)

def sanity_compare(npy_tmp, rows, ref_values, label):
    """Max and mean absolute difference between the new rows and the reference rows. The raise is left to the caller."""
    new = np.load(npy_tmp, mmap_mode="r")
    picked = np.asarray(new[rows], dtype=np.float32)
    del new
    gc.collect()
    assert picked.shape == ref_values.shape, (picked.shape, ref_values.shape)
    diff = np.abs(picked - ref_values)
    result = dict(n_rows=int(len(rows)), max_abs_diff=float(diff.max()), mean_abs_diff=float(diff.mean()), limit=SANITY_LIMIT)
    print(f"SANITY | {label}: {result['n_rows']} rows | max abs diff {result['max_abs_diff']:.3e} | "
          f"mean abs diff {result['mean_abs_diff']:.3e} | limit {SANITY_LIMIT}", flush=True)
    return result

def publish(tmp, final):
    os.replace(tmp, OUT / final)

SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed input of Perch
PERCH_DIM = 1536
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"
NW = int(os.environ.get("M3_WORKERS", "0" if SMOKE else "4"))   # audio-loading workers
PERCH_BATCH = 8 if SMOKE else 64
LOADER_FILES = 8
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 1.5 * 3600))     # whole notebook

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
REF_DIR = find_dir("soundscape_embed_meta.json", "M3_REF_DIR")
ONNX_PATH = Path(os.environ["M3_PERCH_ONNX"]) if os.environ.get("M3_PERCH_ONNX") else find_dir("perch_v2.onnx", "M3_ONNX_DIR") / "perch_v2.onnx"
assert ONNX_PATH.exists(), f"perch_v2.onnx not found: attach strippedbanana/sc4000-birdclef-m3-perch-onnx ({ONNX_PATH})"
print("Data:", DATA, "| reference:", REF_DIR, "| onnx:", ONNX_PATH, "| smoke:", SMOKE)
print("ORT", ort.__version__, ort.get_available_providers())

## File list, reference and model checksum (before any GPU work)

In [ ]:
names = list_soundscapes(DATA)
files = [DATA / "train_soundscapes" / n for n in names]
print(len(files), "soundscapes in train_soundscapes")

ref_values = np.load(REF_DIR / "perch_soundscape_embeddings.npy").astype(np.float32)
if ref_values.ndim != 2 or ref_values.shape[1] != PERCH_DIM:
    raise RuntimeError(f"reference embeddings have shape {ref_values.shape}")
ref_keys = load_reference_keys(REF_DIR / "soundscape_windows.csv", len(ref_values), PERCH_DIM, names)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

t0 = time.time()
perch_sha = sha256_of(ONNX_PATH)
if strict("sha") and perch_sha != PERCH_SHA256:
    raise RuntimeError(f"perch_v2.onnx checksum mismatch: {perch_sha}")
print(f"checksum done in {time.time() - t0:.0f}s")

## Audio code (same decode and failure rules as `ns_a1_all`)

In [ ]:
class EmptyAudioError(Exception):
    pass

class NonFiniteAudioError(Exception):
    pass

# Only these count as a decode failure of the file. Anything else is a bug and stops the notebook.
# soundfile >= 0.11 raises SoundFileError subclasses; older versions raise RuntimeError.
DECODE_ERRORS = (getattr(sf, "SoundFileError", RuntimeError), EmptyAudioError, NonFiniteAudioError)

def to_32k_mono(y, sr):
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def read_windows(path):
    """Whole file: n_win = ceil(len / WIN), end padded with zeros."""
    y, sr = sf.read(str(path), dtype="float32")
    y = to_32k_mono(y, sr)
    if len(y) == 0:
        raise EmptyAudioError(f"no samples in {Path(path).name}")
    if not np.isfinite(y).all():
        raise NonFiniteAudioError(f"non-finite samples in {Path(path).name}")
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_win)])

## Perch 2.0 on the GPU

In [ ]:
GPU_PROVIDERS = [("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"]
PROVIDERS = GPU_PROVIDERS if strict("cuda") else ["CPUExecutionProvider"]

def check_gpu_placement(onnx_path, batch):
    """get_providers() can list CUDA while the graph really runs on CPU, so time each node."""
    so = ort.SessionOptions(); so.enable_profiling = True
    so.profile_file_prefix = str(Path(tempfile.gettempdir()) / "ort_probe")
    s = ort.InferenceSession(str(onnx_path), so, providers=PROVIDERS)
    s.run(["embedding"], {"inputs": batch}); s.run(["embedding"], {"inputs": batch})
    with open(s.end_profiling()) as f:
        events = json.load(f)
    by = {}
    for e in events:
        if e.get("cat") == "Node" and str(e.get("name", "")).endswith("_kernel_time"):
            p = e.get("args", {}).get("provider", "?")
            by[p] = by.get(p, 0) + e.get("dur", 0)
    total = sum(by.values())
    print("Node kernel time by provider (us):", by)
    if not strict("cuda"):
        return
    share = by.get("CUDAExecutionProvider", 0) / total if total else 0.0
    print(f"CUDA share of node time: {share:.1%}")
    if share < 0.5:
        raise RuntimeError(f"Only {share:.1%} of node time is on CUDA: refusing to run on CPU")

class PerchFiles(torch.utils.data.Dataset):
    def __len__(self):
        return len(files)
    def __getitem__(self, j):
        try:
            return j, preprocess_windows(read_windows(files[j])), ""
        except DECODE_ERRORS as e:         # recorded and limited below
            return j, np.zeros((0, WIN), np.float32), repr(e)

def keep_items(items):
    return items

t_perch_start = time.time()
check_gpu_placement(ONNX_PATH, np.zeros((PERCH_BATCH, WIN), np.float32))
print(f"placement probe {time.time() - t_perch_start:.0f}s")
sess = ort.InferenceSession(str(ONNX_PATH), providers=PROVIDERS)
print("Active providers:", sess.get_providers())
if strict("cuda") and sess.get_providers()[0] != "CUDAExecutionProvider":
    raise RuntimeError(f"CUDAExecutionProvider is not active: {sess.get_providers()}")
assert sess.get_inputs()[0].name == "inputs", sess.get_inputs()[0].name
assert "embedding" in [o.name for o in sess.get_outputs()]

loader = torch.utils.data.DataLoader(
    PerchFiles(), batch_size=LOADER_FILES, shuffle=False, num_workers=NW, collate_fn=keep_items,
    prefetch_factor=4 if NW else None)     # numpy goes straight to ORT, so pin_memory has no effect
guard = make_guard("Perch", len(files))
BIN_TMP = OUT / "perch_all_embeddings.tmp.bin"
NPY_TMP = OUT / "perch_all_embeddings.tmp.npy"
row_j, row_w, failed = [], [], []
pending, n_pending, n_done, n_seen, files_ok = [], 0, 0, 0, 0
t_model = 0.0

def run_batches(x, sink):
    global n_done, t_model
    for i in range(0, len(x), PERCH_BATCH):
        t = time.time()
        b = x[i:i + PERCH_BATCH]
        e = sess.run(["embedding"], {"inputs": b})[0]
        t_model += time.time() - t
        assert e.shape == (len(b), PERCH_DIM), e.shape
        sink.write(e.astype("<f4").tobytes())
        n_done += len(b)
        guard(n_done, n_seen, files_ok)

with open(BIN_TMP, "wb") as sink:
    for step, items in enumerate(loader):
        for j, w, err in items:
            if err:
                add_failure(failed, names[j], err); continue
            row_j.extend([j] * len(w)); row_w.extend(range(len(w)))
            pending.append(w); n_pending += len(w); n_seen += len(w); files_ok += 1
        if n_pending >= PERCH_BATCH:
            x = np.concatenate(pending)
            n = len(x) // PERCH_BATCH * PERCH_BATCH
            run_batches(x[:n], sink)
            pending = [x[n:]] if n < len(x) else []; n_pending = len(x) - n
        if step % 25 == 0:
            done = min((step + 1) * LOADER_FILES, len(files)); el = time.time() - t_perch_start
            print(f"Perch {done}/{len(files)} files | {n_done} windows | {el:.0f}s | "
                  f"ETA {el / done * (len(files) - done) / 60:.1f} min | model {t_model:.0f}s", flush=True)
    if pending:
        run_batches(np.concatenate(pending), sink)

perch_wall, perch_model_s = time.time() - t_perch_start, t_model
print(f"Perch done: {n_done} windows | failed decodes {len(failed)} | wall {perch_wall:.0f}s | model {perch_model_s:.0f}s")
del sess, loader
gc.collect()

## Checks against the `soundscape_embed` output, then atomic publication

In [ ]:
windows = pd.DataFrame({"filename": [names[j] for j in row_j], "win": row_w})
N = len(windows)
assert N == n_done == n_seen, (N, n_done, n_seen)
if N == 0:
    raise RuntimeError("no windows were embedded")
check_windows_table(windows, names, failed)

finish_array(BIN_TMP, NPY_TMP, N, PERCH_DIM)
rows = align_to_reference(ref_keys, windows)
sanity = sanity_compare(NPY_TMP, rows, ref_values, "Perch vs soundscape_embed")
if sanity["max_abs_diff"] > SANITY_LIMIT:
    # The temporary array stays in the output for diagnosis; no final name exists, so nothing downstream can use it.
    raise RuntimeError(f"Sanity check failed: max abs diff {sanity['max_abs_diff']:.3e} > {SANITY_LIMIT} "
                       f"(mean {sanity['mean_abs_diff']:.3e}, {sanity['n_rows']} rows)")

per_file = windows.groupby("filename", sort=False).size()
wall_total = time.time() - T_NOTEBOOK
meta = dict(
    perch_sha256=perch_sha, sha_checked=strict("sha"), onnxruntime_version=ort.__version__, device_name=GPU_NAME,
    file_order="sorted(train_soundscapes/*.ogg) by bare filename",
    files_listed=len(names), files_embedded=int(len(per_file)), windows=N,
    windows_per_file_counts={str(int(k)): int(v) for k, v in per_file.value_counts().items()},
    failed_files=[dict(filename=n, error=e) for n, e in failed], sanity_vs_soundscape_embed=sanity,
    wall_s=dict(perch=perch_wall, notebook=wall_total), model_s=dict(perch=perch_model_s))

csv_tmp = OUT / "soundscape_all_windows.tmp.csv"
windows.to_csv(csv_tmp, index=False, lineterminator="\n")
publish(NPY_TMP, "perch_all_embeddings.npy")
publish(csv_tmp, "soundscape_all_windows.csv")
meta_tmp = OUT / "ns_perch_meta.tmp.json"          # written last: its presence means the run is complete
with open(meta_tmp, "w") as f:
    json.dump(meta, f, indent=1)
publish(meta_tmp, "ns_perch_meta.json")

print("Windows per file:", meta["windows_per_file_counts"], "| failed files:", len(failed))
print(f"TIMING | {N} windows, {len(per_file)} files | Perch wall {perch_wall:.0f}s, {N / perch_wall:.1f} windows/s | "
      f"model {perch_model_s:.0f}s | notebook {wall_total:.0f}s = {wall_total / 3600:.2f} h | device {GPU_NAME}")